# Notebook 1: Tiền xử lý dữ liệu cho khai phá luật kết hợp định lượng

Notebook này chuẩn bị dữ liệu cho hai kịch bản thực nghiệm ở Chương 5:

- **Baseline (Post-processing):** chạy FP-Growth trước, sau đó lọc các mẫu có `avg(price) > 20`.
- **Proposed (In-mining Pruning):** cắt tỉa trực tiếp trong quá trình sinh mẫu khi `avg(price) > 20`.

Mục tiêu của notebook này là đọc dataset Market Basket Analysis, làm sạch dữ liệu theo yêu cầu, gom nhóm các mặt hàng theo mã hóa đơn `BillNo`, tạo danh sách giao dịch, tạo bảng giá đại diện cho từng mặt hàng, hiển thị đầy đủ các thống kê cần dùng trong báo cáo, và lưu kết quả tiền xử lý cho các notebook tiếp theo.

## 1. Cấu hình môi trường

Notebook được thiết kế để chạy được trên cả máy local:

- Khi chạy local, notebook tự tìm dataset trong thư mục dự án.
- Khi chạy trên Colab, nếu chưa tìm thấy dataset, notebook sẽ yêu cầu upload file CSV hoặc Excel.
- Có thể đổi đường dẫn dataset bằng biến môi trường `MARKET_BASKET_DATASET_PATH`.
- Có thể đổi thư mục đầu ra bằng biến môi trường `OUTPUT_DIR`.

In [1]:
import importlib.util
import json
import os
import pickle
import subprocess
import sys
from pathlib import Path


def ensure_package(import_name, pip_name=None):
    """Cài thư viện nếu môi trường hiện tại chưa có.

    Khi chạy trong Codex Desktop/local hiện tại, hàm cũng thử dùng thư mục thư viện
    Python đi kèm Codex trước khi gọi pip. Trên Colab/Jupyter thông thường, pandas
    thường đã có sẵn nên bước này không phát sinh cài đặt.
    """
    pip_name = pip_name or import_name
    if importlib.util.find_spec(import_name) is None:
        bundled_packages = Path.home() / ".cache" / "codex-runtimes" / "codex-primary-runtime" / "dependencies" / "python"
        if bundled_packages.exists() and str(bundled_packages) not in sys.path:
            sys.path.insert(0, str(bundled_packages))

    if importlib.util.find_spec(import_name) is None:
        print(f"Thiếu thư viện '{import_name}'. Đang cài đặt gói '{pip_name}'...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pip_name])


ensure_package("pandas")

import pandas as pd

try:
    from IPython.display import display
except Exception:
    def display(obj):
        print(obj)

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

# =========================
# CẤU HÌNH LINH HOẠT
# =========================
# Có thể chỉnh bằng biến môi trường hoặc sửa trực tiếp tại đây khi chạy trên Colab/local.
PROJECT_ROOT = Path(os.environ.get("PROJECT_ROOT", Path.cwd())).resolve()
DATASET_PATH_OVERRIDE = os.environ.get("MARKET_BASKET_DATASET_PATH", "").strip()
OUTPUT_DIR = Path(os.environ.get("OUTPUT_DIR", PROJECT_ROOT / "outputs")).resolve()

# Tên cột được cấu hình để notebook có thể tái sử dụng với dataset cùng cấu trúc nhưng đổi tên file.
TRANSACTION_ID_COLUMN = os.environ.get("TRANSACTION_ID_COLUMN", "BillNo")
ITEM_COLUMN = os.environ.get("ITEM_COLUMN", "Itemname")
PRICE_COLUMN = os.environ.get("PRICE_COLUMN", "Price")
QUANTITY_COLUMN = os.environ.get("QUANTITY_COLUMN", "Quantity")
REQUIRED_COLUMNS = [TRANSACTION_ID_COLUMN, ITEM_COLUMN, PRICE_COLUMN]

# Cách lấy giá đại diện cho từng item: median, mean, min hoặc max.
PRICE_AGG_METHOD = os.environ.get("PRICE_AGG_METHOD", "median").lower()
if PRICE_AGG_METHOD not in {"median", "mean", "min", "max"}:
    raise ValueError("PRICE_AGG_METHOD phải là một trong: median, mean, min, max")

TRANSACTION_ITEM_SEPARATOR = os.environ.get("TRANSACTION_ITEM_SEPARATOR", " | ")
ARTIFACT_FILES = {
    "transactions_pickle": os.environ.get("TRANSACTIONS_PICKLE", "transactions.pkl"),
    "transactions_csv": os.environ.get("TRANSACTIONS_CSV", "transactions.csv"),
    "item_price_pickle": os.environ.get("ITEM_PRICE_PICKLE", "item_price.pkl"),
    "item_price_csv": os.environ.get("ITEM_PRICE_CSV", "item_price.csv"),
    "summary_json": os.environ.get("PREPROCESSING_SUMMARY_JSON", "preprocessing_summary.json"),
}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)
pd.set_option("display.max_colwidth", 80)

print("Môi trường chạy:", "Google Colab" if IN_COLAB else "Local/Jupyter")
print("Thư mục làm việc:", PROJECT_ROOT)
print("Thư mục lưu kết quả:", OUTPUT_DIR)
print("Cột mã giao dịch:", TRANSACTION_ID_COLUMN)
print("Cột tên mặt hàng:", ITEM_COLUMN)
print("Cột giá:", PRICE_COLUMN)
print("Cách lấy giá đại diện:", PRICE_AGG_METHOD)

Môi trường chạy: Local/Jupyter
Thư mục làm việc: D:\CS\DATA_MINING\Constraint-based_pattern_data_mining
Thư mục lưu kết quả: D:\CS\DATA_MINING\Constraint-based_pattern_data_mining\outputs
Cột mã giao dịch: BillNo
Cột tên mặt hàng: Itemname
Cột giá: Price
Cách lấy giá đại diện: median


## 2. Tìm và đọc tập dữ liệu

Dataset được ưu tiên đọc từ file CSV vì tốc độ nhanh hơn. Nếu không có CSV, notebook sẽ đọc file Excel. Notebook tự dò separator phổ biến của CSV (`;`, `,`, tab) và tự chuẩn hóa cột giá, nên không phụ thuộc cứng vào đúng một tên file hay đúng một kiểu dấu thập phân.

In [2]:
SUPPORTED_DATASET_EXTENSIONS = {".csv", ".xlsx", ".xls"}


def infer_csv_separator(dataset_path: Path) -> str:
    """Tự dò separator CSV từ dòng đầu tiên."""
    with dataset_path.open("r", encoding="utf-8-sig", errors="ignore") as f:
        first_line = f.readline()
    candidates = [";", ",", "\t", "|"]
    return max(candidates, key=lambda sep: first_line.count(sep))


def read_preview_columns(dataset_path: Path):
    """Đọc nhanh vài dòng để kiểm tra dataset có đủ cột bắt buộc hay không."""
    suffix = dataset_path.suffix.lower()
    if suffix == ".csv":
        sep = infer_csv_separator(dataset_path)
        preview = pd.read_csv(dataset_path, sep=sep, nrows=5, dtype="string")
    elif suffix in {".xlsx", ".xls"}:
        ensure_package("openpyxl")
        preview = pd.read_excel(dataset_path, nrows=5, dtype="string")
    else:
        return []
    return list(preview.columns)


def looks_like_target_dataset(dataset_path: Path) -> bool:
    """Kiểm tra file có đủ các cột được cấu hình cho bài toán hay không."""
    try:
        columns = read_preview_columns(dataset_path)
    except Exception:
        return False
    return set(REQUIRED_COLUMNS).issubset(set(columns))


def candidate_dataset_files(project_root: Path):
    """Sinh danh sách file dữ liệu ứng viên, không phụ thuộc vào một tên file cố định."""
    if DATASET_PATH_OVERRIDE:
        yield Path(DATASET_PATH_OVERRIDE).expanduser()

    search_roots = [project_root]
    if IN_COLAB:
        search_roots.append(Path("/content"))

    seen = set()
    candidates = []
    for root in search_roots:
        if not root.exists():
            continue
        for suffix in SUPPORTED_DATASET_EXTENSIONS:
            candidates.extend(root.glob(f"*{suffix}"))
            candidates.extend(root.glob(f"*/*{suffix}"))
            candidates.extend(root.glob(f"**/*{suffix}"))

    def sort_key(path):
        suffix_rank = 0 if path.suffix.lower() == ".csv" else 1
        return (suffix_rank, len(path.parts), path.name.lower())

    for path in sorted(candidates, key=sort_key):
        resolved = path.resolve()
        if resolved in seen:
            continue
        seen.add(resolved)
        if OUTPUT_DIR in resolved.parents:
            continue
        yield resolved


def find_dataset(project_root: Path) -> Path:
    """Tìm dataset dựa trên phần mở rộng và bộ cột bắt buộc đã cấu hình."""
    for path in candidate_dataset_files(project_root):
        if path.exists() and path.suffix.lower() in SUPPORTED_DATASET_EXTENSIONS and looks_like_target_dataset(path):
            return path

    if IN_COLAB:
        from google.colab import files  # type: ignore
        print("Không tìm thấy dataset phù hợp. Vui lòng upload file CSV hoặc Excel có đủ các cột đã cấu hình.")
        uploaded = files.upload()
        for filename in uploaded.keys():
            uploaded_path = Path("/content") / filename
            if uploaded_path.suffix.lower() in SUPPORTED_DATASET_EXTENSIONS and looks_like_target_dataset(uploaded_path):
                return uploaded_path

    raise FileNotFoundError(
        "Không tìm thấy dataset phù hợp. Hãy đặt file CSV/Excel có đủ các cột "
        f"{REQUIRED_COLUMNS} trong thư mục dự án, hoặc đặt biến MARKET_BASKET_DATASET_PATH."
    )


def read_market_basket_dataset(dataset_path: Path) -> pd.DataFrame:
    """Đọc dataset Market Basket Analysis từ CSV hoặc Excel."""
    suffix = dataset_path.suffix.lower()
    if suffix == ".csv":
        sep = infer_csv_separator(dataset_path)
        df = pd.read_csv(dataset_path, sep=sep, dtype="string")
    elif suffix in {".xlsx", ".xls"}:
        ensure_package("openpyxl")
        df = pd.read_excel(dataset_path, dtype="string")
    else:
        raise ValueError(f"Định dạng file không được hỗ trợ: {dataset_path.suffix}")

    return df


dataset_path = find_dataset(PROJECT_ROOT)
raw_df = read_market_basket_dataset(dataset_path)

print("Đường dẫn dataset:", dataset_path)
print("Kích thước dữ liệu gốc:", raw_df.shape)
display(raw_df.head(10))

Đường dẫn dataset: D:\CS\DATA_MINING\Constraint-based_pattern_data_mining\data\Assignment-1_Data.csv
Kích thước dữ liệu gốc: (522064, 7)


,BillNo,Itemname,Quantity,Date,Price,CustomerID,Country
0,536365,WHITE HANGING HEART T-LIGHT HOLDER,6,01.12.2010 08:26,"2,55",17850,United Kingdom
1,536365,WHITE METAL LANTERN,6,01.12.2010 08:26,"3,39",17850,United Kingdom
2,536365,CREAM CUPID HEARTS COAT HANGER,8,01.12.2010 08:26,"2,75",17850,United Kingdom
3,536365,KNITTED UNION FLAG HOT WATER BOTTLE,6,01.12.2010 08:26,"3,39",17850,United Kingdom
4,536365,RED WOOLLY HOTTIE WHITE HEART.,6,01.12.2010 08:26,"3,39",17850,United Kingdom
5,536365,SET 7 BABUSHKA NESTING BOXES,2,01.12.2010 08:26,"7,65",17850,United Kingdom
6,536365,GLASS STAR FROSTED T-LIGHT HOLDER,6,01.12.2010 08:26,"4,25",17850,United Kingdom
7,536366,HAND WARMER UNION JACK,6,01.12.2010 08:28,"1,85",17850,United Kingdom
8,536366,HAND WARMER RED POLKA DOT,6,01.12.2010 08:28,"1,85",17850,United Kingdom
9,536367,ASSORTED COLOUR BIRD ORNAMENT,32,01.12.2010 08:34,"1,69",13047,United Kingdom


## 3. Kiểm tra cấu trúc và chất lượng dữ liệu gốc

Bước này hiển thị các cột, kiểu dữ liệu, số lượng giá trị thiếu và một vài thống kê ban đầu. Các thông tin này giúp xác nhận dataset đã được đọc đúng định dạng trước khi làm sạch.

In [3]:
required_columns = REQUIRED_COLUMNS
missing_required_columns = [col for col in required_columns if col not in raw_df.columns]
if missing_required_columns:
    raise KeyError(f"Dataset thiếu các cột bắt buộc: {missing_required_columns}")

# Chuẩn hóa kiểu dữ liệu cho các cột quan trọng.
raw_df[TRANSACTION_ID_COLUMN] = raw_df[TRANSACTION_ID_COLUMN].astype("string")
raw_df[ITEM_COLUMN] = raw_df[ITEM_COLUMN].astype("string")
raw_df[PRICE_COLUMN] = pd.to_numeric(
    raw_df[PRICE_COLUMN].astype("string").str.replace(",", ".", regex=False),
    errors="coerce",
)
if QUANTITY_COLUMN in raw_df.columns:
    raw_df[QUANTITY_COLUMN] = pd.to_numeric(
        raw_df[QUANTITY_COLUMN].astype("string").str.replace(",", ".", regex=False),
        errors="coerce",
    )

schema_df = pd.DataFrame({
    "Cột": raw_df.columns,
    "Kiểu dữ liệu": [str(raw_df[col].dtype) for col in raw_df.columns],
    "Số giá trị thiếu": [int(raw_df[col].isna().sum()) for col in raw_df.columns],
    "Tỷ lệ thiếu (%)": [round(float(raw_df[col].isna().mean() * 100), 4) for col in raw_df.columns],
})

price_describe = raw_df[PRICE_COLUMN].describe().to_frame(name=PRICE_COLUMN).T

print("Thông tin cột và giá trị thiếu:")
display(schema_df)

print(f"Thống kê mô tả ban đầu của cột {PRICE_COLUMN}:")
display(price_describe)

if QUANTITY_COLUMN in raw_df.columns:
    quantity_describe = raw_df[QUANTITY_COLUMN].describe().to_frame(name=QUANTITY_COLUMN).T
    print(f"Thống kê mô tả ban đầu của cột {QUANTITY_COLUMN}:")
    display(quantity_describe)

Thông tin cột và giá trị thiếu:


,Cột,Kiểu dữ liệu,Số giá trị thiếu,Tỷ lệ thiếu (%)
0,BillNo,string,0,0.0000
1,Itemname,string,1455,0.2787
2,Quantity,Int64,0,0.0000
3,Date,string,0,0.0000
4,Price,Float64,0,0.0000
5,CustomerID,string,134041,25.6752
6,Country,string,0,0.0000


Thống kê mô tả ban đầu của cột Price:


,count,mean,std,min,25%,50%,75%,max
Price,522064.0,3.826801,41.900599,-11062.06,1.25,2.08,4.13,13541.33


Thống kê mô tả ban đầu của cột Quantity:


,count,mean,std,min,25%,50%,75%,max
Quantity,522064.0,10.090435,161.110525,-9600.0,1.0,3.0,10.0,80995.0


## 4. Làm sạch dữ liệu

Theo yêu cầu thực nghiệm, notebook sử dụng trực tiếp cột giá niêm yết đã cấu hình, mặc định là `Price`, và áp dụng hai bước làm sạch chính:

1. Xóa các dòng thiếu dữ liệu ở các cột bắt buộc đã cấu hình.
2. Xóa các giao dịch bị lỗi có giá `<= 0`.

Notebook không tự ý loại thêm các dòng khác ngoài phạm vi trên, để kết quả bám sát mô tả thực nghiệm.

In [4]:
raw_row_count = len(raw_df)

missing_required_mask = raw_df[required_columns].isna().any(axis=1)
price_error_mask = raw_df[PRICE_COLUMN] <= 0
price_error_mask = price_error_mask.fillna(False)

clean_df = raw_df.loc[~missing_required_mask].copy()
clean_df = clean_df.loc[clean_df[PRICE_COLUMN] > 0].copy()

# Chuẩn hóa tên mặt hàng: bỏ khoảng trắng thừa ở hai đầu, giữ nguyên chữ hoa/thường của dataset gốc.
clean_df[ITEM_COLUMN] = clean_df[ITEM_COLUMN].str.strip()
clean_df = clean_df.loc[clean_df[ITEM_COLUMN].str.len() > 0].copy()

cleaning_summary = pd.DataFrame([
    {"Chỉ tiêu": "Số dòng dữ liệu gốc", "Giá trị": raw_row_count},
    {"Chỉ tiêu": f"Số dòng thiếu {'/'.join(required_columns)}", "Giá trị": int(missing_required_mask.sum())},
    {"Chỉ tiêu": f"Số dòng có {PRICE_COLUMN} <= 0", "Giá trị": int(price_error_mask.sum())},
    {"Chỉ tiêu": "Số dòng sau làm sạch", "Giá trị": len(clean_df)},
    {"Chỉ tiêu": "Số dòng bị loại", "Giá trị": raw_row_count - len(clean_df)},
])

print("Tóm tắt quá trình làm sạch dữ liệu:")
display(cleaning_summary)

print("Dữ liệu sau làm sạch, 10 dòng đầu:")
display(clean_df.head(10))

Tóm tắt quá trình làm sạch dữ liệu:


,Chỉ tiêu,Giá trị
0,Số dòng dữ liệu gốc,522064
1,Số dòng thiếu BillNo/Itemname/Price,1455
2,Số dòng có Price <= 0,2513
3,Số dòng sau làm sạch,519551
4,Số dòng bị loại,2513


Dữ liệu sau làm sạch, 10 dòng đầu:


,BillNo,Itemname,Quantity,Date,Price,CustomerID,Country
0,536365,WHITE HANGING HEART T-LIGHT HOLDER,6,01.12.2010 08:26,2.55,17850,United Kingdom
1,536365,WHITE METAL LANTERN,6,01.12.2010 08:26,3.39,17850,United Kingdom
2,536365,CREAM CUPID HEARTS COAT HANGER,8,01.12.2010 08:26,2.75,17850,United Kingdom
3,536365,KNITTED UNION FLAG HOT WATER BOTTLE,6,01.12.2010 08:26,3.39,17850,United Kingdom
4,536365,RED WOOLLY HOTTIE WHITE HEART.,6,01.12.2010 08:26,3.39,17850,United Kingdom
5,536365,SET 7 BABUSHKA NESTING BOXES,2,01.12.2010 08:26,7.65,17850,United Kingdom
6,536365,GLASS STAR FROSTED T-LIGHT HOLDER,6,01.12.2010 08:26,4.25,17850,United Kingdom
7,536366,HAND WARMER UNION JACK,6,01.12.2010 08:28,1.85,17850,United Kingdom
8,536366,HAND WARMER RED POLKA DOT,6,01.12.2010 08:28,1.85,17850,United Kingdom
9,536367,ASSORTED COLOUR BIRD ORNAMENT,32,01.12.2010 08:34,1.69,13047,United Kingdom


## 5. Tạo danh sách giao dịch

Các mặt hàng được gom nhóm theo cột mã giao dịch đã cấu hình, mặc định là `BillNo`. Trong mỗi hóa đơn, nếu một mặt hàng xuất hiện nhiều lần thì chỉ giữ một lần để phù hợp với biểu diễn giao dịch dạng tập hợp item trong khai phá luật kết hợp.

In [5]:
transactions_df = (
    clean_df.groupby(TRANSACTION_ID_COLUMN, sort=False)[ITEM_COLUMN]
    .apply(lambda items: sorted(set(items.dropna().astype(str))))
    .reset_index(name="Items")
)
transactions_df["Số item trong giao dịch"] = transactions_df["Items"].apply(len)
transactions_df = transactions_df.loc[transactions_df["Số item trong giao dịch"] > 0].reset_index(drop=True)

transactions = transactions_df["Items"].tolist()

transaction_length_summary = transactions_df["Số item trong giao dịch"].describe().to_frame(name="Số item trong giao dịch").T

print("Tổng số giao dịch sau tiền xử lý:", len(transactions_df))
print("Thống kê số item trong mỗi giao dịch:")
display(transaction_length_summary)

print(f"Ví dụ 10 giao dịch đầu sau khi gom nhóm theo {TRANSACTION_ID_COLUMN}:")
display(transactions_df.head(10))

Tổng số giao dịch sau tiền xử lý: 19559
Thống kê số item trong mỗi giao dịch:


,count,mean,std,min,25%,50%,75%,max
Số item trong giao dịch,19559.0,26.0136,47.303971,1.0,6.0,15.0,29.0,1108.0


Ví dụ 10 giao dịch đầu sau khi gom nhóm theo BillNo:


,BillNo,Items,Số item trong giao dịch
0,536365,"[CREAM CUPID HEARTS COAT HANGER, GLASS STAR FROSTED T-LIGHT HOLDER, KNITTED ...",7
1,536366,"[HAND WARMER RED POLKA DOT, HAND WARMER UNION JACK]",2
2,536367,"[ASSORTED COLOUR BIRD ORNAMENT, BOX OF 6 ASSORTED COLOUR TEASPOONS, BOX OF V...",12
3,536368,"[BLUE COAT RACK PARIS FASHION, JAM MAKING SET WITH JARS, RED COAT RACK PARIS...",4
4,536369,[BATH BUILDING BLOCK WORD],1
5,536370,"[ALARM CLOCK BAKELIKE GREEN, ALARM CLOCK BAKELIKE PINK, ALARM CLOCK BAKELIKE...",20
6,536371,[PAPER CHAIN KIT 50'S CHRISTMAS],1
7,536372,"[HAND WARMER RED POLKA DOT, HAND WARMER UNION JACK]",2
8,536373,"[CREAM CUPID HEARTS COAT HANGER, EDWARDIAN PARASOL RED, GLASS STAR FROSTED T...",16
9,536374,[VICTORIAN SEWING BOX LARGE],1


## 6. Tạo bảng giá đại diện cho từng mặt hàng

Một mặt hàng có thể xuất hiện với nhiều mức giá khác nhau trong dữ liệu thực tế. Để tính `avg(price)` của một itemset một cách ổn định, notebook tạo giá đại diện cho từng item theo cấu hình `PRICE_AGG_METHOD`. Mặc định là **trung vị** vì vẫn sử dụng trực tiếp giá niêm yết thực tế nhưng ít bị ảnh hưởng bởi ngoại lệ hơn so với trung bình.

In [6]:
item_price_stats = (
    clean_df.groupby(ITEM_COLUMN)
    .agg(
        price_min=(PRICE_COLUMN, "min"),
        price_median=(PRICE_COLUMN, "median"),
        price_mean=(PRICE_COLUMN, "mean"),
        price_max=(PRICE_COLUMN, "max"),
        price_nunique=(PRICE_COLUMN, "nunique"),
        row_count=(PRICE_COLUMN, "size"),
        transaction_count=(TRANSACTION_ID_COLUMN, "nunique"),
    )
    .reset_index()
    .sort_values(["transaction_count", ITEM_COLUMN], ascending=[False, True])
    .reset_index(drop=True)
)

representative_price_column = f"price_{PRICE_AGG_METHOD}"
item_price = dict(zip(item_price_stats[ITEM_COLUMN], item_price_stats[representative_price_column]))
items_with_multiple_prices = int((item_price_stats["price_nunique"] > 1).sum())

print("Tổng số mặt hàng duy nhất:", len(item_price_stats))
print("Số mặt hàng có nhiều hơn một mức giá trong dữ liệu:", items_with_multiple_prices)
print(f"Cột giá đại diện đang dùng: {representative_price_column}")
print("Bảng giá đại diện của 15 mặt hàng xuất hiện nhiều nhất:")
display(item_price_stats.head(15))

Tổng số mặt hàng duy nhất: 4006
Số mặt hàng có nhiều hơn một mức giá trong dữ liệu: 3403
Cột giá đại diện đang dùng: price_median
Bảng giá đại diện của 15 mặt hàng xuất hiện nhiều nhất:


,Itemname,price_min,price_median,price_mean,price_max,price_nunique,row_count,transaction_count
0,WHITE HANGING HEART T-LIGHT HOLDER,2.4,2.95,3.226256,6.77,9,2265,2198
1,JUMBO BAG RED RETROSPOT,1.65,2.08,2.491483,4.95,11,2084,2061
2,REGENCY CAKESTAND 3 TIER,4.0,12.75,14.050627,32.04,9,1929,1904
3,PARTY BUNTING,3.75,4.95,5.81213,15.79,14,1676,1655
4,LUNCH BAG RED RETROSPOT,1.45,1.65,2.135484,8.33,8,1570,1541
5,ASSORTED COLOUR BIRD ORNAMENT,1.35,1.69,1.722819,3.19,5,1465,1431
6,SET OF 3 CAKE TINS PANTRY DESIGN,3.39,4.95,5.857081,12.95,8,1360,1346
7,PACK OF 72 RETROSPOT CAKE CASES,0.4,0.55,0.765316,5.0,8,1328,1279
8,LUNCH BAG BLACK SKULL.,1.45,1.65,2.102913,8.33,7,1315,1260
9,NATURAL SLATE HEART CHALKBOARD,2.55,2.95,3.594655,6.95,5,1246,1232


## 7. Tổng hợp số liệu cho báo cáo

Bảng dưới đây cung cấp các con số chính để điền vào mục **3.1. Môi trường và Tập dữ liệu** trong báo cáo.

In [7]:
summary = {
    "dataset_path": str(dataset_path),
    "environment": "Google Colab" if IN_COLAB else "Local/Jupyter",
    "transaction_id_column": TRANSACTION_ID_COLUMN,
    "item_column": ITEM_COLUMN,
    "price_column": PRICE_COLUMN,
    "price_agg_method": PRICE_AGG_METHOD,
    "representative_price_column": representative_price_column,
    "raw_rows": int(raw_row_count),
    "clean_rows": int(len(clean_df)),
    "removed_rows": int(raw_row_count - len(clean_df)),
    "transactions": int(len(transactions_df)),
    "unique_items": int(len(item_price_stats)),
    "avg_items_per_transaction": float(transactions_df["Số item trong giao dịch"].mean()),
    "median_items_per_transaction": float(transactions_df["Số item trong giao dịch"].median()),
    "min_price": float(clean_df[PRICE_COLUMN].min()),
    "median_price": float(clean_df[PRICE_COLUMN].median()),
    "mean_price": float(clean_df[PRICE_COLUMN].mean()),
    "max_price": float(clean_df[PRICE_COLUMN].max()),
    "items_with_multiple_prices": items_with_multiple_prices,
}

summary_display = pd.DataFrame([
    {"Chỉ tiêu": "Tổng số dòng gốc", "Giá trị": summary["raw_rows"]},
    {"Chỉ tiêu": "Tổng số dòng sau tiền xử lý", "Giá trị": summary["clean_rows"]},
    {"Chỉ tiêu": "Tổng số giao dịch sau tiền xử lý", "Giá trị": summary["transactions"]},
    {"Chỉ tiêu": "Tổng số mặt hàng duy nhất", "Giá trị": summary["unique_items"]},
    {"Chỉ tiêu": "Số item trung bình trong mỗi giao dịch", "Giá trị": round(summary["avg_items_per_transaction"], 4)},
    {"Chỉ tiêu": "Trung vị số item trong mỗi giao dịch", "Giá trị": round(summary["median_items_per_transaction"], 4)},
    {"Chỉ tiêu": "Giá nhỏ nhất sau lọc", "Giá trị": summary["min_price"]},
    {"Chỉ tiêu": "Trung vị giá sau lọc", "Giá trị": summary["median_price"]},
    {"Chỉ tiêu": "Giá trung bình sau lọc", "Giá trị": round(summary["mean_price"], 4)},
    {"Chỉ tiêu": "Giá lớn nhất sau lọc", "Giá trị": summary["max_price"]},
])

display(summary_display)

,Chỉ tiêu,Giá trị
0,Tổng số dòng gốc,522064.0000
1,Tổng số dòng sau tiền xử lý,519551.0000
2,Tổng số giao dịch sau tiền xử lý,19559.0000
3,Tổng số mặt hàng duy nhất,4006.0000
4,Số item trung bình trong mỗi giao dịch,26.0136
5,Trung vị số item trong mỗi giao dịch,15.0000
6,Giá nhỏ nhất sau lọc,0.0010
7,Trung vị giá sau lọc,2.0800
8,Giá trung bình sau lọc,3.8879
9,Giá lớn nhất sau lọc,13541.3300


## 8. Lưu dữ liệu tiền xử lý cho các notebook tiếp theo

Các file được lưu trong thư mục `outputs/`:

- File giao dịch dạng pickle: mặc định `transactions.pkl`.
- File giao dịch dạng CSV: mặc định `transactions.csv`.
- File giá item dạng pickle: mặc định `item_price.pkl`.
- File thống kê giá item dạng CSV: mặc định `item_price.csv`.
- File tóm tắt tiền xử lý dạng JSON: mặc định `preprocessing_summary.json`.

Các tên file này có thể đổi trong cell cấu hình đầu notebook hoặc bằng biến môi trường.

In [9]:
transactions_path = OUTPUT_DIR / ARTIFACT_FILES["transactions_pickle"]
transactions_csv_path = OUTPUT_DIR / ARTIFACT_FILES["transactions_csv"]
item_price_path = OUTPUT_DIR / ARTIFACT_FILES["item_price_pickle"]
item_price_csv_path = OUTPUT_DIR / ARTIFACT_FILES["item_price_csv"]
summary_path = OUTPUT_DIR / ARTIFACT_FILES["summary_json"]

with transactions_path.open("wb") as f:
    pickle.dump(transactions, f)

transactions_export_df = transactions_df.copy()
transactions_export_df["Items"] = transactions_export_df["Items"].apply(lambda items: TRANSACTION_ITEM_SEPARATOR.join(items))
transactions_export_df.to_csv(transactions_csv_path, index=False, encoding="utf-8-sig")

with item_price_path.open("wb") as f:
    pickle.dump(item_price, f)

item_price_stats.to_csv(item_price_csv_path, index=False, encoding="utf-8-sig")

with summary_path.open("w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

saved_files_df = pd.DataFrame([
    {"File": str(transactions_path), "Mô tả": "Danh sách giao dịch dạng pickle"},
    {"File": str(transactions_csv_path), "Mô tả": "Danh sách giao dịch dạng CSV"},
    {"File": str(item_price_path), "Mô tả": "Dictionary giá đại diện của từng item"},
    {"File": str(item_price_csv_path), "Mô tả": "Bảng thống kê giá của từng item"},
    {"File": str(summary_path), "Mô tả": "Tóm tắt số liệu tiền xử lý"},
])

print("Đã lưu xong các file tiền xử lý:")
display(saved_files_df)

Đã lưu xong các file tiền xử lý:


,File,Mô tả
0,D:\CS\DATA_MINING\Constraint-based_pattern_data_mining\outputs\transactions.pkl,Danh sách giao dịch dạng pickle
1,D:\CS\DATA_MINING\Constraint-based_pattern_data_mining\outputs\transactions.csv,Danh sách giao dịch dạng CSV
2,D:\CS\DATA_MINING\Constraint-based_pattern_data_mining\outputs\item_price.pkl,Dictionary giá đại diện của từng item
3,D:\CS\DATA_MINING\Constraint-based_pattern_data_mining\outputs\item_price.csv,Bảng thống kê giá của từng item
4,D:\CS\DATA_MINING\Constraint-based_pattern_data_mining\outputs\preprocessing...,Tóm tắt số liệu tiền xử lý
